# 04 — Korolev WBF Regime Comparison

This notebook implements the Korolev and Mazin (2003) formulation to calculate the critical updraft velocities for ice growth ($u_{zi}$) and liquid droplet growth ($u_{zw}$). It then compares the actual model vertical velocity against these thresholds to determine the occurrence of the Wegener-Bergeron-Findeisen (WBF) regime in the tracked plumes, directly comparable to the findings in Omanovic et al. (2024, 2025).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

src_dir = next(p / "src" for p in (Path.cwd(), *Path.cwd().parents)
               if (p / "src" / "polarcap_runtime.py").is_file())
sys.path.insert(0, str(src_dir))

from utilities import load_plume_path_runs

In [ ]:
# --- Configuration ---
PROCESSED_ROOT = Path("../../data/processed")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

SEEDING_START = np.datetime64("2023-01-25T10:28:00")

# Korolev formulation (from Omanovic_etal_2024/korolev.py)
def korolev(T, p, rho_air, qv, Ni, qi, Nw, qw):
    """
    Calculate critical updraft velocities for ice and water.
    T: temperature [K]
    p: pressure [Pa]
    rho_air: air density [kg/m3]
    qv: specific humidity [kg/kg]
    Ni: ice concentration [m-3]
    qi: ice mass mixing ratio [kg/kg]
    Nw: droplet concentration [m-3]
    qw: liquid mass mixing ratio [kg/kg]
    """
    T0 = 273.15  # K
    p0 = 101325  # Pa
    rho_i = 917  # kg/m3
    rho_w = 997  # kg/m3
    Rv = 461.5  # J/kg/K
    Mw = 0.018  # kg/mol, water
    Ma = 0.028  # kg/mol, air
    c = 1  # particle shape, assuming spheres
    g = 9.81  # m/s2

    # Handle zeros to avoid division by zero
    Ni_safe = np.where(Ni > 0, Ni, 1e-10)
    Nw_safe = np.where(Nw > 0, Nw, 1e-10)
    
    mi = qi / (Ni_safe / rho_air)  # mean mass, kg
    mi = np.where(mi > 0, mi, 1e-20)
    voli = mi / rho_i  # ice volume
    ri = ((3 * voli) / (4 * np.pi)) ** (1. / 3.)  # mean ice radius, m
    
    mw = qw / Nw_safe
    mw = np.where(mw > 0, mw, 1e-20)
    volw = mw / rho_w
    rw = ((3 * volw) / (4 * np.pi)) ** (1. / 3.)

    Aw = 2.53e11  # Pa
    Bw = 5420  # K
    e_sw = Aw * np.exp(-(Bw / T))  # Pa

    Ai = 3.64e12  # Pa
    Bi = 6148  # K
    e_si = Ai * np.exp(-(Bi / T))  # Pa

    Rd = 287  # J/kg/K
    cp_d = 1005  # J/kg/K
    omega = (Mw / Ma) * (e_sw / (p - e_sw))
    cp = cp_d + 1884 * omega
    Ra = (1 + 0.608 * qv) * Rd

    k = 4.1868e-3 * (5.69 + 0.017 * (T - T0))  # J/m/s/K
    Dv = 2.11e-5 * (T / T0) ** 1.94 * (p0 / p)  # m2/s
    Lv = (56579 - 42.212 * T + np.exp(0.1149 * (281.6 - T))) * (1 / Mw)  # J/kg
    Ls = (46782.5 + 35.8925 * T - 0.07414 * (T ** 2) + 541.5 * np.exp(-(T / 123.75)) ** 2) * (1 / Mw)  # J/kg

    Aice = 1 / ((rho_i * Ls ** 2) / (k * Rv * (T ** 2)) + (rho_i * Rv * T) / (e_si * Dv))
    Bice = 4 * np.pi / rho_air * rho_i * ((e_sw / e_si) - 1) * c * Aice
    a2 = 1 / qv + (Lv * Ls) / (cp * Rv * (T ** 2))
    bi = a2 * Bice
    a0 = g / (Ra * T) * ((Lv * Ra) / (cp * Rv * T) - 1)

    u_z_i = (bi * Ni * ri) / a0

    Awat = 1 / ((rho_w * Lv ** 2) / (k * Rv * T ** 2) + (rho_w * Rv * T) / (e_sw * Dv))
    Bwat = 4 * np.pi * rho_w * Awat / rho_air
    a1 = 1 / qv + Lv ** 2 / (cp * Rv * T ** 2)
    bw = a1 * Bwat

    u_z_w = ((1 - (e_sw / e_si)) * bw * Nw * rho_air * rw) / ((e_sw / e_si) * a0)

    return u_z_i, u_z_w, ri, rw


In [ ]:
# --- Load model data ---
datasets = load_plume_path_runs(processed_root=PROCESSED_ROOT, kinds=("integrated",))
run_label = list(datasets.keys())[0]
ds = datasets[run_label]["integrated"]
print(f"Run: {run_label}")


In [ ]:
# --- Extract and convert variables to SI units for Korolev formula ---
elapsed_s = (ds.time - SEEDING_START).values / np.timedelta64(1, 's')
elapsed_min = elapsed_s / 60.0

# Ensure consistent dimensions (time, cell)
def extract_2d(var_name, ds):
    if var_name in ds.data_vars:
        if 'diameter' in ds[var_name].dims:
            return ds[var_name].sum(dim='diameter').transpose('time', 'cell').values
        else:
            return ds[var_name].transpose('time', 'cell').values
    return np.zeros((ds.sizes['time'], ds.sizes['cell']))

temp = extract_2d('t', ds)            # K
rho = extract_2d('rho', ds)           # kg/m3
p0 = ds['p0'].values if 'p0' in ds else 100000.0
pp = extract_2d('pp', ds)
pres = p0 + pp                        # Pa
qv = extract_2d('qv', ds)             # kg/kg (assuming qv in model is kg/kg)

# Microphysics
icnc = extract_2d('icnc', ds)         # L-1
cdnc = extract_2d('cdnc', ds)         # cm-3
qi_gl = extract_2d('qi', ds)          # g/L = kg/m3
qc_gm3 = extract_2d('qc', ds)         # g/m3

# Convert to SI
Ni = icnc * 1000.0                    # L-1 -> m-3
Nw = cdnc * 1e6                       # cm-3 -> m-3
qi_kgkg = qi_gl / rho                 # kg/m3 / kg/m3 = kg/kg
qw_kgkg = (qc_gm3 / 1000.0) / rho     # g/m3 -> kg/m3 -> kg/kg

# Vertical velocity
# If wt is extremely large, it might not be in m/s despite attributes, but let's assume it's m/s if scaled, 
# or maybe it's just `wt` from the model which is vertical velocity. 
# If we need w in m/s and it's given in something else, we will just plot it as is for now.
wt = extract_2d('wt', ds)

wi, wc, ri, rw = korolev(temp, pres, rho, qv, Ni, qi_kgkg, Nw, qw_kgkg)


In [ ]:
# --- Classify Regimes ---
# WBF regime: wc < w < wi (Ice grows, water evaporates)
# Mixed-phase growth: w > wi (Both grow)
# Evaporation: w < wc (Both shrink)

is_wbf = (wt > wc) & (wt < wi)
is_growth = (wt >= wi)
is_evap = (wt <= wc)

# Calculate fraction of time in each regime (ensemble average across cells)
frac_wbf = np.mean(is_wbf, axis=1) * 100
frac_growth = np.mean(is_growth, axis=1) * 100
frac_evap = np.mean(is_evap, axis=1) * 100


In [ ]:
# --- Figure: WBF Comparison ---
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Panel A: Critical velocities vs actual velocity (for a representative cell, e.g., cell 0)
cell_idx = 0
ax = axes[0]
ax.plot(elapsed_min, wi[:, cell_idx], 'b-', label='$u_{zi}$ (Critical for ice)', lw=2)
ax.plot(elapsed_min, wc[:, cell_idx], 'r-', label='$u_{zw}$ (Critical for liquid)', lw=2)
ax.plot(elapsed_min, wt[:, cell_idx], 'k--', label='$w$ (Actual updraft)', lw=1.5, alpha=0.7)
ax.fill_between(elapsed_min, wc[:, cell_idx], wi[:, cell_idx], color='purple', alpha=0.2, label='WBF Regime Zone')

ax.set_ylabel('Vertical Velocity (m/s)')
ax.set_title(f'(a) Korolev Updraft Thresholds (Cell {cell_idx})')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
# ax.set_ylim(-1, 5) # Adjust limits as needed depending on wt units

# Panel B: Ensemble Regime Frequency
ax = axes[1]
ax.plot(elapsed_min, frac_growth, 'g-', label='Mixed-Phase Growth ($w > u_{zi}$)', lw=2)
ax.plot(elapsed_min, frac_wbf, 'purple', label='WBF Regime ($u_{zw} < w < u_{zi}$)', lw=2)
ax.plot(elapsed_min, frac_evap, 'orange', label='Evaporation ($w < u_{zw}$)', lw=2)

ax.set_xlabel('Elapsed time (min)')
ax.set_ylabel('Frequency in Regime (%)')
ax.set_title('(b) Plume WBF Regime Evolution (Ensemble Average)')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 30)
ax.set_ylim(-5, 105)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_korolev_wbf_regime.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'figure_korolev_wbf_regime.png'}")
